# Validation:
BOS → predicts A

BOS A → predicts B

BOS A B → predicts C

BOS A B C → predicts D

Where the input is always the true sequence.
# Free generation
BOS → predicts A

BOS A → predicts B

BOS A B → predicts X   ← mistake!!

BOS A B X → predicts Y

BOS A B X Y → predicts Z

At this point it is operating in a distribution it did not see during training. That's the type of behavior we want to diagnose.

Let's look at the different questions the following functions respond to:

- `validate()`: "How good is my next-token prediction?"

- `teacher_forced_reconstruction()`: "What image do my next-token predictions corresponds to?"

- `generate_and_visualize_textures()`: "What happens when my model generates a complete sequence on its own?"

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
from PIL import Image
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.nn.functional as F
import torch
import torch.optim as optim
import matplotlib.pyplot as plt
import time
import numpy as np
import math
import sys

In [3]:
REPO_DIR = '/content/Texture_synthesis'
os.chdir('/content')

if not os.path.exists(REPO_DIR):
  !git clone https://github.com/ValentinaEmili/Texture-synthesis.git Texture_synthesis

if REPO_DIR not in sys.path:
  sys.path.append(REPO_DIR)

Cloning into 'Texture_synthesis'...
remote: Enumerating objects: 318, done.
remote: Counting objects: 100% (318/318), done.
remote: Compressing objects: 100% (298/298), done.
remote: Total 318 (delta 135), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (318/318), 47.59 MiB | 8.43 MiB/s, done.
Resolving deltas: 100% (135/135), done.


In [ ]:
!pip install import-ipynb -q
import import_ipynb

In [4]:
from Texture_synthesis.codebook.VQ_VAE import VQ_VAE
from Texture_synthesis.generation.Transformers.Transformers import Transformer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# VQ-VAE
quant_save_path = '/content/drive/MyDrive/DeepLearning/dtd/checkpoints/vq_vae/epoch_39.pth'
quantizer = VQ_VAE().to(device)
quantizer.load_state_dict(torch.load(quant_save_path, weights_only=True))

transformer = Transformer(num_embeddings=512, seq_len=1024, n_layers=8).to(device)
optimizer = optim.Adam(transformer.parameters(), lr=2e-4, betas=(0.9, 0.999))
save_path = '/content/drive/MyDrive/DeepLearning/dtd/checkpoints/transformers/vqvae_best.pth'
transformer.load_state_dict(torch.load(save_path, weights_only=True))

best_model = "vqvae_best.pth"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 50.0 MB/s eta 0:00:00


In [5]:
from Texture_synthesis.codebook.VQGAN.VQGAN import VQGAN, Discriminator
from Texture_synthesis.generation.Transformers.Transformers import Transformer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# VQ-GAN
quant_save_path = '/content/drive/MyDrive/DeepLearning/dtd/checkpoints/vq_gan/epoch_19/generator/main_model.pth'
quantizer = VQGAN().to(device)
quantizer.load_state_dict(torch.load(quant_save_path, weights_only=True))

transformer = Transformer(num_embeddings=512, seq_len=1024, n_layers=8).to(device)
optimizer = optim.Adam(transformer.parameters(), lr=2e-4, betas=(0.9, 0.999))
save_path = '/content/drive/MyDrive/DeepLearning/dtd/checkpoints/transformers/vqgan_best.pth'
transformer.load_state_dict(torch.load(save_path, weights_only=True))

best_model = 'vqgan_best.pth'

In [6]:
train_transform = transforms.Compose([
        transforms.Resize(512),
        transforms.RandomCrop(512),
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
        ])

eval_transform = transforms.Compose([
        transforms.Resize(512),
        transforms.CenterCrop(512),
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
        ])

In [11]:
from Texture_synthesis.codebook.VQGAN.VQGAN_TT import VQGAN, Discriminator
from Texture_synthesis.generation.Transformers.Transformers import Transformer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# VQ-GAN from Taming Transformers https://github.com/compvis/taming-transformers
quant_save_path = '/content/drive/MyDrive/DeepLearning/dtd/checkpoints/vq_gan/taming_transformers/epoch_19/generator/main_model.pth'
quantizer = VQGAN().to(device)
quantizer.load_state_dict(torch.load(quant_save_path, weights_only=True))

transformer = Transformer(num_embeddings=1024, seq_len=256, n_layers=8).to(device)
optimizer = optim.Adam(transformer.parameters(), lr=2e-4, betas=(0.9, 0.999))
save_path = '/content/drive/MyDrive/DeepLearning/dtd/checkpoints/transformers/vqgan_tt_best.pth'
transformer.load_state_dict(torch.load(save_path, weights_only=True))

best_model = 'vqgan_tt_best.pth'

In [ ]:
# VQ-GAN from Taming Transformers paper

train_transform = transforms.Compose([
        transforms.Resize(256),
        transforms.RandomCrop(256),
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
        ])

eval_transform = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(256),
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
        ])

In [7]:
class DTD_Dataset(Dataset):
    def __init__(self, root, file_list, transform=None, class_to_idx=None):
        self.root = root
        self.transform = transform

        with open(file_list, mode='r', encoding='utf-8') as f:
            self.files = [line.strip() for line in f if line.strip()]

        if class_to_idx is None:
          unique_classes = sorted({os.path.normpath(p).split(os.sep)[0] for p in self.files})
          self.class_to_idx = {class_name: i for i, class_name in enumerate(unique_classes)}
        else:
          self.class_to_idx = class_to_idx

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        relative_path = self.files[idx]
        image_path = os.path.join(self.root, relative_path)
        img = Image.open(image_path).convert('RGB')
        rel_norm = os.path.normpath(relative_path)
        string_label = rel_norm.split(os.sep, 1)[0]
        label = self.class_to_idx[string_label]

        if self.transform:
            img = self.transform(img)

        return img, label

In [8]:
batch_size = 8
path_images = "drive/MyDrive/DeepLearning/dtd/images"
path_labels = "drive/MyDrive/DeepLearning/dtd/labels"
train_dataset = DTD_Dataset(path_images, os.path.join(path_labels, "train1.txt"), train_transform)
class_to_idx = train_dataset.class_to_idx
val_dataset = DTD_Dataset(path_images, os.path.join(path_labels, "val1.txt"), eval_transform, class_to_idx=class_to_idx)
test_dataset = DTD_Dataset(path_images, os.path.join(path_labels, "test1.txt"), eval_transform, class_to_idx=class_to_idx)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

In [9]:
@torch.no_grad()
def extract_codebook_indices(quantizer, data):
  quantizer.eval()

  embedding_dim = quantizer.vq.embedding_dim
  embeddings = quantizer.vq.embeddings.weight                               # (num_embeddings, embed_dim)

  z = quantizer.encoder(data)                                               # (batch_size, embed_dim, h, w)
  z_flattened = z.permute(0, 2, 3, 1).contiguous().view(-1, embedding_dim)  # (batch_size * h * w, embed_dim)

  distances = (torch.sum(z_flattened**2, dim=1, keepdim=True)
                    + torch.sum(embeddings**2, dim=1)
                    - 2 * torch.matmul(z_flattened, embeddings.t()))        # (batch_size * h * w, num_embeddings)

  indices = torch.argmin(distances, dim=1)        # (batch_size * h * w)
  return indices.reshape(z.shape[0],-1)           # (batch_size, seq_len)

# Training and validation

In [ ]:
@torch.no_grad()
def validate(val_loader, transformer, quantizer, device):
  transformer.eval()
  quantizer.eval()

  transformer.to(device)
  quantizer.to(device)

  bos_token_id = transformer.num_embeddings

  total_loss, total_accuracy, total_tokens = 0.0, 0.0, 0.0

  for batch_idx, (data, _) in enumerate(val_loader):
    data = data.to(device)

    indices = extract_codebook_indices(quantizer, data)
    b, t = indices.shape                      # (batch_size, seq_len)

    bos_tokens = torch.full((b, 1), bos_token_id, dtype=torch.long, device=device)
    full_sequence = torch.cat([bos_tokens, indices], dim=1) # (batch_size, seq_len)

    inputs = full_sequence[:, :-1]            # [BOS, c1, c2, ..., c_{t-1}]
    targets = full_sequence[:, 1:]            # [c1, c2, ..., c_{t-1}, c_{t}]

    num_tokens = targets.numel()
    total_tokens += num_tokens

    logits = transformer(inputs)              # (batch_size, seq_len, num_embeddings)

    predictions = torch.argmax(logits, dim=-1)
    accuracy = (predictions == targets).sum().item()

    targets = targets.reshape(-1)                             # (batch_size * seq_len)
    logits = logits.reshape(-1, transformer.num_embeddings) # (batch_size * seq_len, num_embeddings)

    loss = F.cross_entropy(logits, targets, reduction="sum")

    total_loss += loss.item()
    total_accuracy += accuracy

  mean_loss = total_loss / total_tokens
  mean_accuracy = (total_accuracy / total_tokens) * 100
  return mean_loss, mean_accuracy

In [ ]:
def train_and_validation(train_loader, val_loader, transformer, quantizer, optimizer, device, epochs=20):
  transformer.to(device)
  quantizer.to(device)

  bos_token_id = transformer.num_embeddings
  os.makedirs(save_path, exist_ok=True)

  best_val_loss = float('inf')

  for epoch in range(epochs):
    transformer.train()
    quantizer.eval()
    total_loss, total_accuracy, total_tokens = 0.0, 0.0, 0

    for batch_idx, (data, _) in enumerate(train_loader):
      data = data.to(device)

      indices = extract_codebook_indices(quantizer, data)
      b, t = indices.shape                      # (batch_size, seq_len)

      bos_tokens = torch.full((b, 1), bos_token_id, dtype=torch.long, device=device)
      full_sequence = torch.cat([bos_tokens, indices], dim=1) # (batch_size, seq_len)

      inputs = full_sequence[:, :-1]            # [BOS, c1, c2, ..., c_{t-1}]
      targets = full_sequence[:, 1:]            # [c1, c2, ..., c_{t-1}, c_{t}]

      num_tokens = targets.numel()
      total_tokens += num_tokens

      logits = transformer(inputs)              # (batch_size, seq_len, num_embeddings)

      predictions = torch.argmax(logits, dim=-1)
      accuracy = (predictions == targets).sum().item()

      optimizer.zero_grad()

      targets = targets.reshape(-1)                             # (batch_size * seq_len)
      logits = logits.reshape(-1, transformer.num_embeddings) # (batch_size * seq_len, num_embeddings)

      loss = F.cross_entropy(logits, targets)

      loss.backward()
      optimizer.step()

      total_loss += loss.item() * num_tokens
      total_accuracy += accuracy

    train_loss = total_loss / total_tokens
    train_accuracy = (total_accuracy / total_tokens) * 100
    train_perplexity = np.exp(train_loss)

    print(f"====> Epoch {epoch} Finished")
    print(f"====> Train Loss: {train_loss:.4f} | Train Accuracy: {train_accuracy:.4f} | Train Perplexity: {train_perplexity:.2f}")

    epoch_save_path = os.path.join(save_path, f"epoch_{epoch}.pth")
    torch.save(transformer.state_dict(), epoch_save_path)

    val_loss, val_accuracy = validate(val_loader, transformer, quantizer, device)
    val_perplexity = np.exp(val_loss)
    print(f"====> Valid Loss: {val_loss:.4f} | Valid Accuracy: {val_accuracy:.4f} | Valid Perplexity: {val_perplexity:.2f}\n")

    if val_loss < best_val_loss:
      best_val_loss = val_loss
      best_save_path = os.path.join(save_path, best_model)
      torch.save(transformer.state_dict(), best_save_path)
      print(f"New best model at epoch {epoch}")

In [ ]:
train_and_validation(train_loader, val_loader, transformer, quantizer, optimizer, device, epochs=50)

## Results

### VQ-VAE + Transformer

====> Epoch 0 Finished
====> Train Loss: 2.9539 | Train Accuracy: 20.7475 | Train Perplexity: 19.18
====> Valid Loss: 2.9315 | Valid Accuracy: 20.9745 | Valid Perplexity: 18.76

====> Epoch 1 Finished
====> Train Loss: 2.9243 | Train Accuracy: 21.0155 | Train Perplexity: 18.62
====> Valid Loss: 2.9087 | Valid Accuracy: 21.2102 | Valid Perplexity: 18.33

====> Epoch 2 Finished
====> Train Loss: 2.8993 | Train Accuracy: 21.3581 | Train Perplexity: 18.16
====> Valid Loss: 2.8891 | Valid Accuracy: 21.4460 | Valid Perplexity: 17.98

====> Epoch 3 Finished
====> Train Loss: 2.8800 | Train Accuracy: 21.6265 | Train Perplexity: 17.81
====> Valid Loss: 2.8780 | Valid Accuracy: 21.5571 | Valid Perplexity: 17.78

====> Epoch 4 Finished
====> Train Loss: 2.8656 | Train Accuracy: 21.8437 | Train Perplexity: 17.56
====> Valid Loss: 2.8574 | Valid Accuracy: 21.9316 | Valid Perplexity: 17.42

====> Epoch 5 Finished
====> Train Loss: 2.8469 | Train Accuracy: 22.1015 | Train Perplexity: 17.23
====> Valid Loss: 2.8461 | Valid Accuracy: 21.9594 | Valid Perplexity: 17.22

====> Epoch 6 Finished
====> Train Loss: 2.8344 | Train Accuracy: 22.2701 | Train Perplexity: 17.02
====> Valid Loss: 2.8390 | Valid Accuracy: 22.2148 | Valid Perplexity: 17.10

====> Epoch 7 Finished
====> Train Loss: 2.8181 | Train Accuracy: 22.6029 | Train Perplexity: 16.75
====> Valid Loss: 2.8164 | Valid Accuracy: 22.5352 | Valid Perplexity: 16.72

====> Epoch 8 Finished
====> Train Loss: 2.8055 | Train Accuracy: 22.7680 | Train Perplexity: 16.53
====> Valid Loss: 2.8200 | Valid Accuracy: 22.3903 | Valid Perplexity: 16.78

====> Epoch 9 Finished
====> Train Loss: 2.7922 | Train Accuracy: 23.0873 | Train Perplexity: 16.32
====> Valid Loss: 2.8038 | Valid Accuracy: 22.7483 | Valid Perplexity: 16.51

====> Epoch 10 Finished
====> Train Loss: 2.7816 | Train Accuracy: 23.3238 | Train Perplexity: 16.15
====> Valid Loss: 2.7942 | Valid Accuracy: 22.9322 | Valid Perplexity: 16.35

====> Epoch 11 Finished
====> Train Loss: 2.7677 | Train Accuracy: 23.5647 | Train Perplexity: 15.92
====> Valid Loss: 2.7804 | Valid Accuracy: 23.2048 | Valid Perplexity: 16.13

====> Epoch 12 Finished
====> Train Loss: 2.7543 | Train Accuracy: 23.8546 | Train Perplexity: 15.71
====> Valid Loss: 2.7641 | Valid Accuracy: 23.4714 | Valid Perplexity: 15.86

====> Epoch 13 Finished
====> Train Loss: 2.7382 | Train Accuracy: 24.0923 | Train Perplexity: 15.46
====> Valid Loss: 2.7559 | Valid Accuracy: 23.7134 | Valid Perplexity: 15.74

====> Epoch 14 Finished
====> Train Loss: 2.7273 | Train Accuracy: 24.2595 | Train Perplexity: 15.29
====> Valid Loss: 2.7394 | Valid Accuracy: 23.9726 | Valid Perplexity: 15.48

====> Epoch 15 Finished
====> Train Loss: 2.7113 | Train Accuracy: 24.6752 | Train Perplexity: 15.05
====> Valid Loss: 2.7264 | Valid Accuracy: 24.2413 | Valid Perplexity: 15.28

====> Epoch 16 Finished
====> Train Loss: 2.6969 | Train Accuracy: 24.9474 | Train Perplexity: 14.83
====> Valid Loss: 2.7168 | Valid Accuracy: 24.4095 | Valid Perplexity: 15.13

====> Epoch 17 Finished
====> Train Loss: 2.6883 | Train Accuracy: 25.1084 | Train Perplexity: 14.71
====> Valid Loss: 2.7002 | Valid Accuracy: 24.6620 | Valid Perplexity: 14.88

====> Epoch 18 Finished
====> Train Loss: 2.6721 | Train Accuracy: 25.4544 | Train Perplexity: 14.47
====> Valid Loss: 2.6781 | Valid Accuracy: 25.2312 | Valid Perplexity: 14.56

====> Epoch 19 Finished
====> Train Loss: 2.6562 | Train Accuracy: 25.8302 | Train Perplexity: 14.24
====> Valid Loss: 2.6491 | Valid Accuracy: 26.0336 | Valid Perplexity: 14.14

====> Epoch 20 Finished
====> Train Loss: 2.6287 | Train Accuracy: 26.5776 | Train Perplexity: 13.86
====> Valid Loss: 2.6266 | Valid Accuracy: 26.6744 | Valid Perplexity: 13.83

====> Epoch 21 Finished
====> Train Loss: 2.6071 | Train Accuracy: 27.0549 | Train Perplexity: 13.56
====> Valid Loss: 2.6015 | Valid Accuracy: 27.1991 | Valid Perplexity: 13.48

====> Epoch 22 Finished
====> Train Loss: 2.5864 | Train Accuracy: 27.6212 | Train Perplexity: 13.28
====> Valid Loss: 2.5864 | Valid Accuracy: 27.5229 | Valid Perplexity: 13.28

====> Epoch 23 Finished
====> Train Loss: 2.5704 | Train Accuracy: 27.8900 | Train Perplexity: 13.07
====> Valid Loss: 2.5696 | Valid Accuracy: 27.8907 | Valid Perplexity: 13.06

====> Epoch 24 Finished
====> Train Loss: 2.5575 | Train Accuracy: 28.2342 | Train Perplexity: 12.90
====> Valid Loss: 2.5610 | Valid Accuracy: 28.0726 | Valid Perplexity: 12.95

====> Epoch 25 Finished
====> Train Loss: 2.5424 | Train Accuracy: 28.5073 | Train Perplexity: 12.71
====> Valid Loss: 2.5593 | Valid Accuracy: 28.1603 | Valid Perplexity: 12.93

====> Epoch 26 Finished
====> Train Loss: 2.5322 | Train Accuracy: 28.7181 | Train Perplexity: 12.58
====> Valid Loss: 2.5499 | Valid Accuracy: 28.3222 | Valid Perplexity: 12.81

====> Epoch 27 Finished
====> Train Loss: 2.5210 | Train Accuracy: 29.0125 | Train Perplexity: 12.44
====> Valid Loss: 2.5413 | Valid Accuracy: 28.4126 | Valid Perplexity: 12.70

====> Epoch 28 Finished
====> Train Loss: 2.5126 | Train Accuracy: 29.1853 | Train Perplexity: 12.34
====> Valid Loss: 2.5371 | Valid Accuracy: 28.4990 | Valid Perplexity: 12.64

====> Epoch 29 Finished
====> Train Loss: 2.5014 | Train Accuracy: 29.3816 | Train Perplexity: 12.20
====> Valid Loss: 2.5202 | Valid Accuracy: 28.9070 | Valid Perplexity: 12.43

====> Epoch 30 Finished
====> Train Loss: 2.4930 | Train Accuracy: 29.5685 | Train Perplexity: 12.10
====> Valid Loss: 2.5146 | Valid Accuracy: 29.0700 | Valid Perplexity: 12.36

====> Epoch 31 Finished
====> Train Loss: 2.4822 | Train Accuracy: 29.7913 | Train Perplexity: 11.97
====> Valid Loss: 2.5080 | Valid Accuracy: 29.1689 | Valid Perplexity: 12.28

====> Epoch 32 Finished
====> Train Loss: 2.4720 | Train Accuracy: 30.0302 | Train Perplexity: 11.85
====> Valid Loss: 2.4948 | Valid Accuracy: 29.4661 | Valid Perplexity: 12.12

====> Epoch 33 Finished
====> Train Loss: 2.4625 | Train Accuracy: 30.2534 | Train Perplexity: 11.73
====> Valid Loss: 2.4921 | Valid Accuracy: 29.5652 | Valid Perplexity: 12.09

====> Epoch 34 Finished
====> Train Loss: 2.4523 | Train Accuracy: 30.4503 | Train Perplexity: 11.62
====> Valid Loss: 2.4777 | Valid Accuracy: 29.9458 | Valid Perplexity: 11.91

====> Epoch 35 Finished
====> Train Loss: 2.4383 | Train Accuracy: 30.7743 | Train Perplexity: 11.45
====> Valid Loss: 2.4648 | Valid Accuracy: 30.1554 | Valid Perplexity: 11.76

====> Epoch 36 Finished
====> Train Loss: 2.4279 | Train Accuracy: 31.0449 | Train Perplexity: 11.34
====> Valid Loss: 2.4547 | Valid Accuracy: 30.3888 | Valid Perplexity: 11.64

====> Epoch 37 Finished
====> Train Loss: 2.4195 | Train Accuracy: 31.2345 | Train Perplexity: 11.24
====> Valid Loss: 2.4533 | Valid Accuracy: 30.3969 | Valid Perplexity: 11.63

====> Epoch 38 Finished
====> Train Loss: 2.4129 | Train Accuracy: 31.3761 | Train Perplexity: 11.17
====> Valid Loss: 2.4399 | Valid Accuracy: 30.7186 | Valid Perplexity: 11.47

====> Epoch 39 Finished
====> Train Loss: 2.4003 | Train Accuracy: 31.6615 | Train Perplexity: 11.03
====> Valid Loss: 2.4393 | Valid Accuracy: 30.6954 | Valid Perplexity: 11.46

====> Epoch 40 Finished
====> Train Loss: 2.3918 | Train Accuracy: 31.8543 | Train Perplexity: 10.93
====> Valid Loss: 2.4312 | Valid Accuracy: 30.9214 | Valid Perplexity: 11.37

====> Epoch 41 Finished
====> Train Loss: 2.3856 | Train Accuracy: 31.9440 | Train Perplexity: 10.87
====> Valid Loss: 2.4256 | Valid Accuracy: 31.0886 | Valid Perplexity: 11.31

====> Epoch 42 Finished
====> Train Loss: 2.3780 | Train Accuracy: 32.1440 | Train Perplexity: 10.78
====> Valid Loss: 2.4149 | Valid Accuracy: 31.2872 | Valid Perplexity: 11.19

====> Epoch 43 Finished
====> Train Loss: 2.3701 | Train Accuracy: 32.2960 | Train Perplexity: 10.70
====> Valid Loss: 2.4076 | Valid Accuracy: 31.4658 | Valid Perplexity: 11.11

====> Epoch 44 Finished
====> Train Loss: 2.3621 | Train Accuracy: 32.5255 | Train Perplexity: 10.61
====> Valid Loss: 2.4061 | Valid Accuracy: 31.5449 | Valid Perplexity: 11.09

====> Epoch 45 Finished
====> Train Loss: 2.3534 | Train Accuracy: 32.7436 | Train Perplexity: 10.52
====> Valid Loss: 2.4007 | Valid Accuracy: 31.6352 | Valid Perplexity: 11.03

====> Epoch 46 Finished
====> Train Loss: 2.3476 | Train Accuracy: 32.8305 | Train Perplexity: 10.46
====> Valid Loss: 2.3944 | Valid Accuracy: 31.8401 | Valid Perplexity: 10.96

====> Epoch 47 Finished
====> Train Loss: 2.3380 | Train Accuracy: 33.0504 | Train Perplexity: 10.36
====> Valid Loss: 2.3865 | Valid Accuracy: 31.9876 | Valid Perplexity: 10.88

====> Epoch 48 Finished
====> Train Loss: 2.3327 | Train Accuracy: 33.1890 | Train Perplexity: 10.31
====> Valid Loss: 2.3852 | Valid Accuracy: 32.0231 | Valid Perplexity: 10.86

====> Epoch 49 Finished
====> Train Loss: 2.3248 | Train Accuracy: 33.3592 | Train Perplexity: 10.22
====> Valid Loss: 2.3829 | Valid Accuracy: 32.0682 | Valid Perplexity: 10.84

New best model at epoch 49

### VQ-GAN + Transformer

====> Epoch 0 Finished
====> Train Loss: 2.3600 | Train Accuracy: 43.0022 | Train Perplexity: 10.59
====> Valid Loss: 1.8969 | Valid Accuracy: 46.5600 | Valid Perplexity: 6.67

====> Epoch 1 Finished
====> Train Loss: 1.8930 | Train Accuracy: 45.9097 | Train Perplexity: 6.64
====> Valid Loss: 1.8121 | Valid Accuracy: 47.4485 | Valid Perplexity: 6.12

====> Epoch 2 Finished
====> Train Loss: 1.8318 | Train Accuracy: 46.6556 | Train Perplexity: 6.24
====> Valid Loss: 1.7868 | Valid Accuracy: 47.7529 | Valid Perplexity: 5.97

====> Epoch 3 Finished
====> Train Loss: 1.7998 | Train Accuracy: 47.2077 | Train Perplexity: 6.05
====> Valid Loss: 1.7573 | Valid Accuracy: 48.6339 | Valid Perplexity: 5.80

====> Epoch 4 Finished
====> Train Loss: 1.7751 | Train Accuracy: 47.7568 | Train Perplexity: 5.90
====> Valid Loss: 1.7250 | Valid Accuracy: 49.3787 | Valid Perplexity: 5.61

====> Epoch 5 Finished
====> Train Loss: 1.7391 | Train Accuracy: 48.4913 | Train Perplexity: 5.69
====> Valid Loss: 1.6785 | Valid Accuracy: 50.1914 | Valid Perplexity: 5.36

====> Epoch 6 Finished
====> Train Loss: 1.7012 | Train Accuracy: 49.2341 | Train Perplexity: 5.48
====> Valid Loss: 1.6492 | Valid Accuracy: 50.7821 | Valid Perplexity: 5.20

====> Epoch 7 Finished
====> Train Loss: 1.6745 | Train Accuracy: 49.7324 | Train Perplexity: 5.34
====> Valid Loss: 1.6294 | Valid Accuracy: 51.0784 | Valid Perplexity: 5.10

====> Epoch 8 Finished
====> Train Loss: 1.6571 | Train Accuracy: 50.1326 | Train Perplexity: 5.24
====> Valid Loss: 1.6140 | Valid Accuracy: 51.5085 | Valid Perplexity: 5.02

====> Epoch 9 Finished
====> Train Loss: 1.6520 | Train Accuracy: 50.1735 | Train Perplexity: 5.22
====> Valid Loss: 1.6047 | Valid Accuracy: 51.7956 | Valid Perplexity: 4.98

====> Epoch 10 Finished
====> Train Loss: 1.6390 | Train Accuracy: 50.4521 | Train Perplexity: 5.15
====> Valid Loss: 1.5969 | Valid Accuracy: 51.8245 | Valid Perplexity: 4.94

====> Epoch 11 Finished
====> Train Loss: 1.6349 | Train Accuracy: 50.4904 | Train Perplexity: 5.13
====> Valid Loss: 1.5911 | Valid Accuracy: 51.9835 | Valid Perplexity: 4.91

====> Epoch 12 Finished
====> Train Loss: 1.6278 | Train Accuracy: 50.6707 | Train Perplexity: 5.09
====> Valid Loss: 1.5836 | Valid Accuracy: 52.1065 | Valid Perplexity: 4.87

====> Epoch 13 Finished
====> Train Loss: 1.6174 | Train Accuracy: 50.9797 | Train Perplexity: 5.04
====> Valid Loss: 1.5864 | Valid Accuracy: 52.0508 | Valid Perplexity: 4.89

====> Epoch 14 Finished
====> Train Loss: 1.6135 | Train Accuracy: 51.0075 | Train Perplexity: 5.02
====> Valid Loss: 1.5745 | Valid Accuracy: 52.3597 | Valid Perplexity: 4.83

====> Epoch 15 Finished
====> Train Loss: 1.6106 | Train Accuracy: 51.1033 | Train Perplexity: 5.01
====> Valid Loss: 1.5807 | Valid Accuracy: 52.1611 | Valid Perplexity: 4.86

====> Epoch 16 Finished
====> Train Loss: 1.6078 | Train Accuracy: 51.1719 | Train Perplexity: 4.99
====> Valid Loss: 1.5732 | Valid Accuracy: 52.4618 | Valid Perplexity: 4.82

====> Epoch 17 Finished
====> Train Loss: 1.6015 | Train Accuracy: 51.3146 | Train Perplexity: 4.96
====> Valid Loss: 1.5692 | Valid Accuracy: 52.5443 | Valid Perplexity: 4.80

====> Epoch 18 Finished
====> Train Loss: 1.5956 | Train Accuracy: 51.4844 | Train Perplexity: 4.93
====> Valid Loss: 1.5678 | Valid Accuracy: 52.3999 | Valid Perplexity: 4.80

====> Epoch 19 Finished
====> Train Loss: 1.5932 | Train Accuracy: 51.4927 | Train Perplexity: 4.92
====> Valid Loss: 1.5588 | Valid Accuracy: 52.7263 | Valid Perplexity: 4.75

====> Epoch 20 Finished
====> Train Loss: 1.5892 | Train Accuracy: 51.6032 | Train Perplexity: 4.90
====> Valid Loss: 1.5585 | Valid Accuracy: 52.6835 | Valid Perplexity: 4.75

====> Epoch 21 Finished
====> Train Loss: 1.5854 | Train Accuracy: 51.7221 | Train Perplexity: 4.88
====> Valid Loss: 1.5571 | Valid Accuracy: 52.7718 | Valid Perplexity: 4.75

====> Epoch 22 Finished
====> Train Loss: 1.5816 | Train Accuracy: 51.7884 | Train Perplexity: 4.86
====> Valid Loss: 1.5508 | Valid Accuracy: 52.8989 | Valid Perplexity: 4.72

====> Epoch 23 Finished
====> Train Loss: 1.5781 | Train Accuracy: 51.9492 | Train Perplexity: 4.85
====> Valid Loss: 1.5497 | Valid Accuracy: 52.9014 | Valid Perplexity: 4.71

====> Epoch 24 Finished
====> Train Loss: 1.5752 | Train Accuracy: 51.9670 | Train Perplexity: 4.83
====> Valid Loss: 1.5471 | Valid Accuracy: 53.1148 | Valid Perplexity: 4.70

====> Epoch 25 Finished
====> Train Loss: 1.5697 | Train Accuracy: 52.1420 | Train Perplexity: 4.81
====> Valid Loss: 1.5481 | Valid Accuracy: 52.9893 | Valid Perplexity: 4.70

====> Epoch 26 Finished
====> Train Loss: 1.5662 | Train Accuracy: 52.1983 | Train Perplexity: 4.79
====> Valid Loss: 1.5486 | Valid Accuracy: 53.0861 | Valid Perplexity: 4.70

====> Epoch 27 Finished
====> Train Loss: 1.5597 | Train Accuracy: 52.4065 | Train Perplexity: 4.76
====> Valid Loss: 1.5405 | Valid Accuracy: 53.2505 | Valid Perplexity: 4.67

====> Epoch 28 Finished
====> Train Loss: 1.5607 | Train Accuracy: 52.3481 | Train Perplexity: 4.76
====> Valid Loss: 1.5390 | Valid Accuracy: 53.1828 | Valid Perplexity: 4.66

====> Epoch 29 Finished
====> Train Loss: 1.5570 | Train Accuracy: 52.4965 | Train Perplexity: 4.74
====> Valid Loss: 1.5430 | Valid Accuracy: 53.0583 | Valid Perplexity: 4.68

====> Epoch 30 Finished
====> Train Loss: 1.5530 | Train Accuracy: 52.5549 | Train Perplexity: 4.73
====> Valid Loss: 1.5389 | Valid Accuracy: 53.2908 | Valid Perplexity: 4.66

====> Epoch 31 Finished
====> Train Loss: 1.5536 | Train Accuracy: 52.5706 | Train Perplexity: 4.73
====> Valid Loss: 1.5295 | Valid Accuracy: 53.5268 | Valid Perplexity: 4.62

====> Epoch 32 Finished
====> Train Loss: 1.5511 | Train Accuracy: 52.5052 | Train Perplexity: 4.72
====> Valid Loss: 1.5321 | Valid Accuracy: 53.4232 | Valid Perplexity: 4.63

====> Epoch 33 Finished
====> Train Loss: 1.5456 | Train Accuracy: 52.7899 | Train Perplexity: 4.69
====> Valid Loss: 1.5297 | Valid Accuracy: 53.5379 | Valid Perplexity: 4.62

====> Epoch 34 Finished
====> Train Loss: 1.5461 | Train Accuracy: 52.7695 | Train Perplexity: 4.69
====> Valid Loss: 1.5261 | Valid Accuracy: 53.5701 | Valid Perplexity: 4.60

====> Epoch 35 Finished
====> Train Loss: 1.5404 | Train Accuracy: 52.8534 | Train Perplexity: 4.67
====> Valid Loss: 1.5327 | Valid Accuracy: 53.5158 | Valid Perplexity: 4.63

====> Epoch 36 Finished
====> Train Loss: 1.5368 | Train Accuracy: 53.0014 | Train Perplexity: 4.65
====> Valid Loss: 1.5305 | Valid Accuracy: 53.5854 | Valid Perplexity: 4.62

====> Epoch 37 Finished
====> Train Loss: 1.5329 | Train Accuracy: 52.9960 | Train Perplexity: 4.63
====> Valid Loss: 1.5226 | Valid Accuracy: 53.5869 | Valid Perplexity: 4.58

====> Epoch 38 Finished
====> Train Loss: 1.5311 | Train Accuracy: 53.0691 | Train Perplexity: 4.62
====> Valid Loss: 1.5214 | Valid Accuracy: 53.6767 | Valid Perplexity: 4.58

====> Epoch 39 Finished
====> Train Loss: 1.5304 | Train Accuracy: 53.1427 | Train Perplexity: 4.62
====> Valid Loss: 1.5223 | Valid Accuracy: 53.7161 | Valid Perplexity: 4.58

====> Epoch 40 Finished
====> Train Loss: 1.5250 | Train Accuracy: 53.3768 | Train Perplexity: 4.59
====> Valid Loss: 1.5180 | Valid Accuracy: 53.8414 | Valid Perplexity: 4.56

====> Epoch 41 Finished
====> Train Loss: 1.5235 | Train Accuracy: 53.4190 | Train Perplexity: 4.59
====> Valid Loss: 1.5187 | Valid Accuracy: 53.8477 | Valid Perplexity: 4.57

====> Epoch 42 Finished
====> Train Loss: 1.5187 | Train Accuracy: 53.5217 | Train Perplexity: 4.57
====> Valid Loss: 1.5175 | Valid Accuracy: 53.9183 | Valid Perplexity: 4.56

====> Epoch 43 Finished
====> Train Loss: 1.5166 | Train Accuracy: 53.4489 | Train Perplexity: 4.56
====> Valid Loss: 1.5145 | Valid Accuracy: 53.8510 | Valid Perplexity: 4.55

====> Epoch 44 Finished
====> Train Loss: 1.5153 | Train Accuracy: 53.6006 | Train Perplexity: 4.55
====> Valid Loss: 1.5177 | Valid Accuracy: 53.8643 | Valid Perplexity: 4.56

====> Epoch 45 Finished
====> Train Loss: 1.5099 | Train Accuracy: 53.7544 | Train Perplexity: 4.53
====> Valid Loss: 1.5133 | Valid Accuracy: 53.8616 | Valid Perplexity: 4.54

====> Epoch 46 Finished
====> Train Loss: 1.5079 | Train Accuracy: 53.7872 | Train Perplexity: 4.52
====> Valid Loss: 1.5155 | Valid Accuracy: 53.9052 | Valid Perplexity: 4.55

====> Epoch 47 Finished
====> Train Loss: 1.5047 | Train Accuracy: 53.8784 | Train Perplexity: 4.50
====> Valid Loss: 1.5067 | Valid Accuracy: 54.1645 | Valid Perplexity: 4.51

====> Epoch 48 Finished
====> Train Loss: 1.5013 | Train Accuracy: 53.9281 | Train Perplexity: 4.49
====> Valid Loss: 1.5156 | Valid Accuracy: 54.0012 | Valid Perplexity: 4.55

====> Epoch 49 Finished
====> Train Loss: 1.5007 | Train Accuracy: 53.9200 | Train Perplexity: 4.48
====> Valid Loss: 1.5057 | Valid Accuracy: 54.1298 | Valid Perplexity: 4.51

New best model at epoch 49

### VQ-GAN (Taming Transformers) + Transformer

# Generation

In [ ]:
@torch.no_grad()
def generate_and_visualize_textures(transformer, quantizer, device, batch_size=4, seq_len=1024, temperature=0.8, top_k=None, grid_size=(1, 4)):
  transformer.eval()
  quantizer.eval()

  transformer.to(device)
  quantizer.to(device)

  h_lat, w_lat = 32, 32 # 32x32=1024
  embeddings = quantizer.vq.embeddings.weight
  embed_dim = quantizer.vq.embedding_dim
  bos_token_id = transformer.num_embeddings
  sequence = torch.full((batch_size, 1), bos_token_id, dtype=torch.long, device=device)

  with torch.no_grad():
    for _ in range(seq_len):
      logits = transformer(sequence)
      next_token_logits = logits[:, -1, :]                          # (batch_size, num_embeddings)

      next_token_logits /= temperature

      if top_k is not None:
        top_k = min(top_k, next_token_logits.shape[1])
        highest_scores, _ = torch.topk(next_token_logits, top_k, dim=-1)
        threshold = highest_scores[:, -1:]
        next_token_logits[next_token_logits < threshold] = -float('Inf')


      probs = F.softmax(next_token_logits, dim=-1)
      next_token = torch.multinomial(probs, 1)
      sequence = torch.cat([sequence, next_token], dim=1)

    final_indices = sequence[:, 1:]
    z = embeddings[final_indices]
    z = z.view(batch_size, h_lat, w_lat, embed_dim).permute(0, 3, 1, 2).contiguous()
    generated_img = quantizer.decoder(z)
    generated_img = (generated_img + 1.0) / 2.0 # [-1, 1] -> [0, 1]
    generated_img = generated_img.permute(0, 2, 3, 1).cpu()

    rows, cols = grid_size
    figs, axes = plt.subplots(rows, cols, figsize=(cols * 4, rows * 4))
    axes = axes.flatten()

    num_displayed_imgs = min(len(generated_img), len(axes))

    for i in range(num_displayed_imgs):
      axes[i].imshow(generated_img[i])
      axes[i].axis('off')

    plt.show()

In [ ]:
temperatures = [0.5, 0.7, 0.8, 1.0, 1.2]
top_ks = [10, 20, 50, 100]

for temperature in temperatures:
  for top_k in top_ks:
    print(f"Temp: {temperature} | Top-{top_k}")
    generate_and_visualize_textures(transformer, quantizer, device, temperature=temperature, top_k=top_k)


# Teacher forcing
A training strategy used in the development of sequence-to-sequence models, by providing the correct input at each step of the sequence rather than allowing the model to generate the next step based on its previous outputs.

In [ ]:
@torch.no_grad()
def teacher_forced_reconstruction(val_loader, transformer, quantizer, device, data):
  transformer.eval()
  quantizer.eval()

  transformer.to(device)
  quantizer.to(device)

  indices = extract_codebook_indices(quantizer, data)
  b, t = indices.shape

  bos_token_id = transformer.num_embeddings
  bos_tokens = torch.full((b, 1), bos_token_id, dtype=torch.long, device=device)
  full_sequence = torch.cat([bos_tokens, indices], dim=1)

  inputs = full_sequence[:, :-1]
  targets = full_sequence[:, 1:]

  logits = transformer(inputs)

  predictions = torch.argmax(logits, dim=-1)

  accuracy = (predictions == targets).float().mean().item() * 100
  logits = logits.reshape(-1, transformer.num_embeddings)
  targets = targets.reshape(-1)
  loss = F.cross_entropy(logits, targets).item()
  perplexity = np.exp(loss).item()

  print(f"Teacher-forced loss: {loss:.4f}")
  print(f"Teacher-forced accuracy: {accuracy:.2f}")
  print(f"Teacher-forced perplexity: {perplexity:.2f}")

  h_lat, w_lat = 32, 32
  embeddings = quantizer.vq.embeddings.weight
  embed_dim = quantizer.vq.embedding_dim

  z = embeddings[predictions]
  z = z.view(b, h_lat, w_lat, embed_dim).permute(0, 3, 1, 2).contiguous()
  reconstructed = quantizer.decoder(z)
  reconstructed = (reconstructed + 1.0) / 2.0 # [-1, 1] -> [0, 1]
  reconstructed = reconstructed.permute(0, 2, 3, 1).cpu()

  original = data.cpu()
  original = (original + 1.0) / 2.0
  original = original.permute(0, 2, 3, 1)

  fig, axes = plt.subplots(2, batch_size, figsize=(batch_size * 4, 8))

  for i in range(batch_size):
    axes[0, i].imshow(original[i].clamp(0, 1))
    axes[0, i].set_title(f"Original {i+1}")
    axes[0, i].axis("off")

    axes[1, i].imshow(reconstructed[i].clamp(0, 1))
    axes[1, i].set_title(f"Transformer TF {i+1}")
    axes[1, i].axis("off")

  plt.show()

In [ ]:
for batch_idx, (data, _) in enumerate(val_loader):
  data = data.to(device)
  teacher_forced_reconstruction(val_loader, transformer, quantizer,device, data)
  if batch_idx == 5: break